# Worksheet 02 Solution

This notebook completes the **Image Classification Using Softmax Regression** worksheet.

It includes:

- Softmax helper functions and test cases
- Loss, cost, gradient, and gradient descent implementations
- Dataset loading, visualization, training, and evaluation
- Linear separability and logistic regression plots
- Written answers for Question 1, Question 2, and Question 3

The notebook first looks for a local MNIST CSV file. If it is not available, it falls back to the `scikit-learn` digits dataset upsampled to `28 x 28` so the notebook remains runnable.

For very large CSV files, the notebook samples a manageable subset by default so training finishes in notebook time. You can disable that by setting `max_samples=None` in the dataset-loading cell.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True


## Helper Functions

We begin by implementing the helper functions required for softmax regression.


In [ ]:
def softmax(z):
    """
    Compute the softmax probabilities for a given input matrix.

    Parameters:
    z (numpy.ndarray): Logits of shape (m, n) or (n,).

    Returns:
    numpy.ndarray: Softmax probability matrix with rows summing to 1.
    """
    z = np.asarray(z, dtype=np.float64)
    was_one_dimensional = z.ndim == 1

    if was_one_dimensional:
        z = z.reshape(1, -1)

    z_stable = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z_stable)
    probabilities = exp_z / np.sum(exp_z, axis=1, keepdims=True)

    return probabilities[0] if was_one_dimensional else probabilities


In [ ]:
z_test = np.array([[2.0, 1.0, 0.1], [1.0, 1.0, 1.0]])
softmax_output = softmax(z_test)

row_sums = np.sum(softmax_output, axis=1)
assert np.allclose(row_sums, 1), f"Test failed: Row sums are {row_sums}"

print("Softmax function passed the test case!")


In [ ]:
def predict_softmax(X, W, b):
    """
    Predict class labels for a set of samples using a trained softmax model.
    """
    logits = np.dot(X, W) + b
    probabilities = softmax(logits)
    predicted_classes = np.argmax(probabilities, axis=1)
    return predicted_classes


In [ ]:
X_test = np.array([[0.2, 0.8], [0.5, 0.5], [0.9, 0.1]])
W_test = np.array([[0.4, 0.2, 0.1], [0.3, 0.7, 0.5]])
b_test = np.array([0.1, 0.2, 0.3])

y_pred_test = predict_softmax(X_test, W_test, b_test)
assert y_pred_test.shape == (3,), f"Test failed: Expected shape (3,), got {y_pred_test.shape}"

print("Predicted class labels:", y_pred_test)


In [ ]:
def loss_softmax(y_pred, y):
    """
    Compute categorical cross-entropy loss.
    Works for a single sample or a full batch.
    """
    y_pred = np.clip(np.asarray(y_pred, dtype=np.float64), 1e-15, 1.0)
    y = np.asarray(y, dtype=np.float64)

    if y_pred.ndim == 1:
        return -np.sum(y * np.log(y_pred))

    return -np.sum(y * np.log(y_pred)) / y.shape[0]


def cost_softmax(X, y, W, b):
    """
    Compute the average softmax regression cost over all samples.
    """
    n = X.shape[0]
    logits = np.dot(X, W) + b
    y_pred = softmax(logits)
    total_loss = -np.sum(y * np.log(np.clip(y_pred, 1e-15, 1.0)))
    return total_loss / n


In [ ]:
y_true_correct = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]])
y_pred_correct = np.array([
    [0.9, 0.05, 0.05],
    [0.1, 0.85, 0.05],
    [0.05, 0.1, 0.85],
])

y_pred_incorrect = np.array([
    [0.05, 0.05, 0.9],
    [0.1, 0.05, 0.85],
    [0.85, 0.1, 0.05],
])

loss_correct = loss_softmax(y_pred_correct, y_true_correct)
loss_incorrect = loss_softmax(y_pred_incorrect, y_true_correct)
assert loss_correct < loss_incorrect, (
    f"Test failed: Expected loss_correct < loss_incorrect, but got "
    f"{loss_correct:.4f} >= {loss_incorrect:.4f}"
)

print(f"Cross-Entropy Loss (Correct Predictions): {loss_correct:.4f}")
print(f"Cross-Entropy Loss (Incorrect Predictions): {loss_incorrect:.4f}")

X_correct = np.array([[1.0, 0.0], [0.0, 1.0]])
y_correct = np.array([[1, 0], [0, 1]])
W_correct = np.array([[5.0, -2.0], [-3.0, 5.0]])
b_correct = np.array([0.1, 0.1])

X_incorrect = np.array([[0.1, 0.9], [0.8, 0.2]])
y_incorrect = np.array([[1, 0], [0, 1]])
W_incorrect = np.array([[0.1, 2.0], [1.5, 0.3]])
b_incorrect = np.array([0.5, 0.6])

cost_correct = cost_softmax(X_correct, y_correct, W_correct, b_correct)
cost_incorrect = cost_softmax(X_incorrect, y_incorrect, W_incorrect, b_incorrect)
assert cost_incorrect > cost_correct, (
    f"Test failed: Incorrect cost {cost_incorrect} is not greater than correct cost {cost_correct}"
)

print("Cost for correct prediction:", cost_correct)
print("Cost for incorrect prediction:", cost_incorrect)
print("Cost function passed the test case!")


In [ ]:
def compute_gradient_softmax(X, y, W, b):
    """
    Compute gradients of the average cross-entropy cost with respect to W and b.
    """
    n = X.shape[0]
    logits = np.dot(X, W) + b
    y_pred = softmax(logits)
    error = y_pred - y

    grad_W = np.dot(X.T, error) / n
    grad_b = np.sum(error, axis=0) / n

    return grad_W, grad_b


In [ ]:
X_test = np.array([[0.2, 0.8], [0.5, 0.5], [0.9, 0.1]])
y_test = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]])
W_test = np.array([[0.4, 0.2, 0.1], [0.3, 0.7, 0.5]])
b_test = np.array([0.1, 0.2, 0.3])

grad_W, grad_b = compute_gradient_softmax(X_test, y_test, W_test, b_test)

z_test = np.dot(X_test, W_test) + b_test
y_pred_test = softmax(z_test)
grad_W_manual = np.dot(X_test.T, (y_pred_test - y_test)) / X_test.shape[0]
grad_b_manual = np.sum(y_pred_test - y_test, axis=0) / X_test.shape[0]

assert np.allclose(grad_W, grad_W_manual), (
    f"Test failed: Gradients w.r.t. W are not equal.\nExpected: {grad_W_manual}\nGot: {grad_W}"
)
assert np.allclose(grad_b, grad_b_manual), (
    f"Test failed: Gradients w.r.t. b are not equal.\nExpected: {grad_b_manual}\nGot: {grad_b}"
)

print("Gradient w.r.t. W:", grad_W)
print("Gradient w.r.t. b:", grad_b)
print("Gradient function passed the test case!")


In [ ]:
def gradient_descent_softmax(X, y, W, b, alpha, n_iter, show_cost=False):
    """
    Perform batch gradient descent for softmax regression.
    """
    W = W.copy()
    b = b.copy()
    cost_history = []

    for i in range(n_iter):
        grad_W, grad_b = compute_gradient_softmax(X, y, W, b)

        W -= alpha * grad_W
        b -= alpha * grad_b

        cost = cost_softmax(X, y, W, b)
        cost_history.append(cost)

        if show_cost and (i % 50 == 0 or i == n_iter - 1):
            print(f"Iteration {i + 1:>4}/{n_iter}: cost = {cost:.4f}")

    return W, b, cost_history


## Preparing the Dataset

**Question 1: Is extracting pixel values sufficient for effective feature extraction? Why or why not?**

Raw pixel values are a reasonable **baseline** for MNIST because the digits are already centered, normalized, grayscale, and visually simple. In that controlled setting, a linear or softmax model can still learn useful patterns from pixel intensities alone.

However, raw pixels are **not generally sufficient** for stronger image understanding. They do not explicitly capture edges, stroke structure, translation or rotation invariance, or higher-level patterns. They are also sensitive to noise and small shifts in the input. So for MNIST they work well enough as a simple baseline, but for more complex image tasks we usually need richer feature extraction or more expressive models such as CNNs.


In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split


def resolve_mnist_csv(csv_file=None):
    candidates = []

    if csv_file is not None:
        candidates.append(Path(csv_file).expanduser())

    candidates.extend(
        [
            Path("mnist_dataset_2025.csv"),
            Path("dataset/mnist_dataset.csv"),
            Path("dataset/mnist_train.csv"),
            Path.cwd() / "mnist_dataset_2025.csv",
            Path.cwd() / "dataset" / "mnist_dataset.csv",
            Path.cwd() / "dataset" / "mnist_train.csv",
            Path("/Users/r5sman/Downloads/mnist_dataset_2025.csv"),
        ]
    )

    for pattern in ("*mnist*.csv", "*MNIST*.csv"):
        candidates.extend(sorted(Path.cwd().glob(pattern)))
        candidates.extend(sorted(Path("/Users/r5sman/Downloads").glob(pattern)))

    seen = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if candidate.exists():
            return candidate

    return None


def build_digits_fallback_dataframe():
    digits = load_digits()
    images_8x8 = digits.images.astype(np.float64)

    # Upsample 8x8 images to 32x32 and crop to 28x28 to mimic MNIST's shape.
    images_32x32 = np.kron(images_8x8, np.ones((4, 4)))
    images_28x28 = images_32x32[:, 2:30, 2:30]
    pixel_values = (images_28x28 * (255.0 / 16.0)).reshape(len(images_28x28), -1)

    df = pd.DataFrame(pixel_values)
    df.insert(0, "label", digits.target.astype(int))
    return df


def plot_sample_images(X, y):
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))

    for ax, digit in zip(axes.ravel(), range(10)):
        matches = np.where(y == digit)[0]
        if len(matches) == 0:
            ax.set_title(f"Digit: {digit}")
            ax.axis("off")
            continue

        image = X[matches[0]].reshape(28, 28)
        ax.imshow(image, cmap="gray")
        ax.set_title(f"Digit: {digit}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()


def load_and_prepare_mnist(csv_file=None, test_size=0.2, random_state=42, max_samples=12000):
    csv_path = resolve_mnist_csv(csv_file)

    if csv_path is not None:
        df = pd.read_csv(csv_path)
        data_source = f"Local CSV: {csv_path}"
    else:
        df = build_digits_fallback_dataframe()
        data_source = (
            "Fallback dataset: scikit-learn digits upsampled to 28x28 "
            "because no local MNIST CSV was found"
        )

    if max_samples is not None and len(df) > max_samples:
        df = df.sample(n=max_samples, random_state=random_state)
        sample_note = f"Sampled {max_samples} rows from the full dataset for faster training."
    else:
        sample_note = "Using the full available dataset."

    y = df.iloc[:, 0].to_numpy(dtype=int)
    X = df.iloc[:, 1:].to_numpy(dtype=np.float64)
    X = X / 255.0

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y,
    )

    print(f"Dataset source: {data_source}")
    print(f"Dataset shape: {X.shape}")
    print(sample_note)
    plot_sample_images(X, y)

    return X_train, X_test, y_train, y_test, data_source


In [ ]:
csv_file_path = "mnist_dataset_2025.csv"
X_train, X_test, y_train, y_test, data_source = load_and_prepare_mnist(
    csv_file_path,
    max_samples=12000,
)

assert len(X_train) == len(y_train), (
    f"Error: X and y have different lengths! X={len(X_train)}, y={len(y_train)}"
)
print("Move forward: Dimension of feature matrix X and label vector y matched.")


## Training the Softmax Regression Model

The code below one-hot encodes the labels, initializes the parameters, trains the model with batch gradient descent, and plots the loss curve.


In [ ]:
from sklearn.preprocessing import OneHotEncoder

np.random.seed(42)

try:
    encoder = OneHotEncoder(sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(sparse=False)

y_train_encoded = encoder.fit_transform(y_train.reshape(-1, 1))
y_test_encoded = encoder.transform(y_test.reshape(-1, 1))

d = X_train.shape[1]
c = y_train_encoded.shape[1]

W = np.random.randn(d, c) * 0.01
b = np.zeros(c)

if X_train.shape[0] > 5000:
    alpha = 0.4
    n_iter = 150
else:
    alpha = 0.3
    n_iter = 300

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Number of classes: {c}")
print(f"Learning rate: {alpha}")
print(f"Iterations: {n_iter}")

W_opt, b_opt, cost_history = gradient_descent_softmax(
    X_train,
    y_train_encoded,
    W,
    b,
    alpha,
    n_iter,
    show_cost=True,
)

plt.figure(figsize=(8, 5))
plt.plot(cost_history, color="navy")
plt.title("Cost Function vs. Iterations")
plt.xlabel("Iterations")
plt.ylabel("Cost")
plt.grid(True)
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score


def evaluate_classification(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    accuracy = np.mean(y_true == y_pred)
    precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    return cm, accuracy, precision, recall, f1


y_pred_test = predict_softmax(X_test, W_opt, b_opt)
cm, accuracy, precision, recall, f1 = evaluate_classification(y_test, y_pred_test)

print("Dataset source used for training:", data_source)
print("\nConfusion Matrix:")
print(cm)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

fig, ax = plt.subplots(figsize=(10, 8))
cax = ax.imshow(cm, cmap="Blues")

num_classes = cm.shape[0]
ax.set_xticks(range(num_classes))
ax.set_yticks(range(num_classes))
ax.set_xticklabels([f"Pred {i}" for i in range(num_classes)])
ax.set_yticklabels([f"True {i}" for i in range(num_classes)])

threshold = np.max(cm) / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center",
            color="white" if cm[i, j] > threshold else "black",
        )

ax.grid(False)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.colorbar(cax)
plt.show()


## Linear Separability and Logistic Regression

The following code reproduces the worksheet's linear vs. non-linear separability experiment.


In [ ]:
from sklearn.datasets import make_classification, make_circles
from sklearn.linear_model import LogisticRegression

np.random.seed(42)

X_linear_separable, y_linear_separable = make_classification(
    n_samples=200,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    random_state=42,
)

X_train_linear, X_test_linear, y_train_linear, y_test_linear = train_test_split(
    X_linear_separable,
    y_linear_separable,
    test_size=0.2,
    random_state=42,
)

logistic_model_linear_separable = LogisticRegression()
logistic_model_linear_separable.fit(X_train_linear, y_train_linear)

X_non_linear_separable, y_non_linear_separable = make_circles(
    n_samples=200,
    noise=0.1,
    factor=0.5,
    random_state=42,
)

X_train_non_linear, X_test_non_linear, y_train_non_linear, y_test_non_linear = train_test_split(
    X_non_linear_separable,
    y_non_linear_separable,
    test_size=0.2,
    random_state=42,
)

logistic_model_non_linear_separable = LogisticRegression()
logistic_model_non_linear_separable.fit(X_train_non_linear, y_train_non_linear)


def plot_decision_boundary(ax, model, X, y, title):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, h),
        np.arange(y_min, y_max, h),
    )
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.8, cmap=plt.cm.Paired)
    ax.scatter(X[:, 0], X[:, 1], c=y, edgecolors="k", cmap=plt.cm.Paired)
    ax.set_title(title)
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")


fig, axes = plt.subplots(2, 2, figsize=(12, 10))

plot_decision_boundary(
    axes[0, 0],
    logistic_model_linear_separable,
    X_train_linear,
    y_train_linear,
    "Linearly Separable Data (Training)",
)
plot_decision_boundary(
    axes[0, 1],
    logistic_model_linear_separable,
    X_test_linear,
    y_test_linear,
    "Linearly Separable Data (Testing)",
)
plot_decision_boundary(
    axes[1, 0],
    logistic_model_non_linear_separable,
    X_train_non_linear,
    y_train_non_linear,
    "Non-Linearly Separable Data (Training)",
)
plot_decision_boundary(
    axes[1, 1],
    logistic_model_non_linear_separable,
    X_test_non_linear,
    y_test_non_linear,
    "Non-Linearly Separable Data (Testing)",
)

plt.tight_layout()
plt.savefig("decision_boundaries.png")
plt.show()


## Written Answers

**Question 2: Provide an interpretation of the output based on your understanding.**

The logistic regression model performs well on the linearly separable dataset because a single straight decision boundary is enough to divide the two classes. That is why the training and testing plots show a clean split with most points on the correct side.

On the non-linearly separable circles dataset, the model struggles because logistic regression is still a linear classifier. It can only learn one linear boundary, while the true structure is circular. As a result, the boundary cuts across the rings and cannot properly separate the inner circle from the outer circle. This shows that model choice must match the geometry of the data.

**Question 3: Describe any challenges you faced while implementing the code above.**

The main challenges were:

- making the softmax implementation numerically stable by subtracting the row-wise maximum before exponentiating
- keeping matrix dimensions consistent across logits, probabilities, gradients, and one-hot encoded labels
- handling the dataset path cleanly, since the worksheet expects a local MNIST CSV but that file may not always be present
- choosing reasonable learning-rate and iteration values so gradient descent converges without making the notebook too slow to run

These issues are common in practical machine learning work, so solving them was part of making the notebook robust rather than only mathematically correct.
